In [1]:
from pathlib import Path
import shutil

# ----------------------------
# File type definition
# ----------------------------

# Unstructured = reports + images only
UNSTRUCTURED_EXT = {
    ".pdf", ".doc", ".docx", ".rtf",
    ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".gif", ".svg",
    ".ppt", ".pptx"
}

# ----------------------------
# Helper: Analyze folder content
# ----------------------------

def analyze_folder(folder: Path):
    extensions = []
    
    for file in folder.rglob("*"):
        if file.is_file():
            extensions.append(file.suffix.lower())
    
    if not extensions:
        return None, None   # empty folder
    
    unique_ext = set(extensions)
    
    # Decide structured vs unstructured by majority
    unstructured_count = sum(ext in UNSTRUCTURED_EXT for ext in extensions)
    structured_count = len(extensions) - unstructured_count
    
    category = "unstructured" if unstructured_count > structured_count else "structured"
    
    # Decide rename rule
    rename_to = None
    if len(unique_ext) == 1:
        rename_to = unique_ext.pop().replace(".", "")  # ".las" → "las"
    
    return category, rename_to


# ----------------------------
# Sorting logic inside one target folder
# ----------------------------

def sort_inside_folder(target_folder: Path):
    
    structured_dir = target_folder / "structured"
    unstructured_dir = target_folder / "unstructured"
    
    structured_dir.mkdir(exist_ok=True)
    unstructured_dir.mkdir(exist_ok=True)
    
    # Handle subfolders
    for item in list(target_folder.iterdir()):
        if item.name in ["structured", "unstructured"]:
            continue
        
        if item.is_dir():
            category, rename_to = analyze_folder(item)
            if category is None:
                continue
            
            destination_root = structured_dir if category == "structured" else unstructured_dir
            
            final_name = rename_to if rename_to else item.name
            destination = destination_root / final_name
            
            if destination.exists():
                final_name = final_name + "_extra"
                destination = destination_root / final_name
            
            shutil.move(str(item), destination)
    
    # Handle loose files directly (no temp folder)
    for file in list(target_folder.iterdir()):
        if file.is_file():
            ext = file.suffix.lower()
            if ext in UNSTRUCTURED_EXT:
                destination = unstructured_dir / file.name
            else:
                destination = structured_dir / file.name
            
            shutil.move(str(file), destination)


# ----------------------------
# Main controller
# ----------------------------

def smart_sort_directory(parent_path, batch_wells_mode=False):
    parent_path = Path(parent_path).resolve()
    
    if not parent_path.exists():
        raise FileNotFoundError(parent_path)
    
    # Batch wells mode → go one level inside
    if batch_wells_mode:
        for well_folder in parent_path.iterdir():
            if well_folder.is_dir():
                print(f"Sorting inside: {well_folder.name}")
                sort_inside_folder(well_folder)
        print("✅ Batch well sorting complete.")
    
    # Single dataset mode
    else:
        print(f"Sorting inside: {parent_path.name}")
        sort_inside_folder(parent_path)
        print("✅ Single dataset sorting complete.")


Single dataset

In [5]:
smart_sort_directory("/Users/apple/Downloads/welldata/SCHOONEBEEK-3091")

Sorting inside: SCHOONEBEEK-3091
✅ Single dataset sorting complete.


Batch dataset

In [2]:
smart_sort_directory("/Users/apple/Downloads/welldata/untitled folder", batch_wells_mode=True)

Sorting inside: SCHOONEBEEK-1402
Sorting inside: SCHOONEBEEK-1601
Sorting inside: SCHOONEBEEK-1392
Sorting inside: SCHOONEBEEK-1501
Sorting inside: SCHOONEBEEK-1302
Sorting inside: SCHOONEBEEK-1401
Sorting inside: SCHOONEBEEK-1602
Sorting inside: SCHOONEBEEK-1651
Sorting inside: SCHOONEBEEK-1391
Sorting inside: SCHOONEBEEK-1505
Sorting inside: SCHOONEBEEK-1502
Sorting inside: SCHOONEBEEK-1301
Sorting inside: SCHOONEBEEK-1503
Sorting inside: SCHOONEBEEK-1504
✅ Batch well sorting complete.
